# 第8章 文埋め込み

## 8.4 最近傍探索ライブラリ`Faiss`を用いた検索

### 8.4.2 `Faiss`を利用した最近傍探索の実装

#### 準備

In [1]:
!pip install 'datasets<4.0.0' faiss-cpu scipy 'transformers[ja,torch]<4.41.0' 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 162.4 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 18.9 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 26.1 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 107.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.9/694.9 kB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 MB 40.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━

#### データセットの読み込みと前処理

In [2]:
from datasets import load_dataset

# Hugging Face Hubのllm-book/jawiki-paragraphsのリポジトリから
# Wikipediaの段落テキストのデータを読み込む
paragraph_dataset = load_dataset(
    "llm-book/jawiki-paragraphs", split="train"
)

README.md: 0.00B [00:00, ?B/s]

jawiki-paragraphs.py: 0.00B [00:00, ?B/s]

default/train/0000.parquet:   0%|          | 0.00/258M [00:00<?, ?B/s]

default/train/0001.parquet:   0%|          | 0.00/260M [00:00<?, ?B/s]

default/train/0002.parquet:   0%|          | 0.00/263M [00:00<?, ?B/s]

...

default/train/0005.parquet:   0%|          | 0.00/258M [00:00<?, ?B/s]

default/train/0006.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

default/train/0007.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

default/train/0008.parquet:   0%|          | 0.00/196M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9668476 [00:00<?, ? examples/s]

In [3]:
# 段落データの形式と事例数を確認する
print(paragraph_dataset)

Dataset({
    features: ['id', 'pageid', 'revid', 'paragraph_index', 'title', 'section', 'text', 'html_tag'],
    num_rows: 9668476
})


In [4]:
from pprint import pprint

# 段落データの内容を確認する
pprint(paragraph_dataset[0])
pprint(paragraph_dataset[1])

{'html_tag': 'p',
 'id': '5-89167474-0',
 'pageid': 5,
 'paragraph_index': 0,
 'revid': 89167474,
 'section': '__LEAD__',
 'text': 'アンパサンド(&, 英語: '
         'ampersand)は、並立助詞「...と...」を意味する記号である。ラテン語で「...と...」を表す接続詞 "et" '
         'の合字を起源とする。現代のフォントでも、Trebuchet MS など一部のフォントでは、"et" '
         'の合字であることが容易にわかる字形を使用している。',
 'title': 'アンパサンド'}
{'html_tag': 'p',
 'id': '5-89167474-1',
 'pageid': 5,
 'paragraph_index': 1,
 'revid': 89167474,
 'section': '語源',
 'text': '英語で教育を行う学校でアルファベットを復唱する場合、その文字自体が単語となる文字("A", "I", かつては "O" '
         'も)については、伝統的にラテン語の per se(それ自体)を用いて "A per se A" '
         'のように唱えられていた。また、アルファベットの最後に、27番目の文字のように "&" を加えることも広く行われていた。"&" '
         'はラテン語で et と読まれていたが、後に英語で and と読まれるようになった。結果として、アルファベットの復唱の最後は "X, Y, '
         'Z, and per se and" という形になった。この最後のフレーズが繰り返されるうちに "ampersand" '
         'と訛っていき、この言葉は1837年までには英語の一般的な語法となった。',
 'title': 'アンパサンド'}


In [5]:
# 段落データのうち、各記事の最初の段落のみ使うようにする
paragraph_dataset = paragraph_dataset.filter(
    lambda example: example["paragraph_index"] == 0
)

Filter:   0%|          | 0/9668476 [00:00<?, ? examples/s]

In [6]:
# フィルタリング後の段落データの形式と事例数を確認する
print(paragraph_dataset)

Dataset({
    features: ['id', 'pageid', 'revid', 'paragraph_index', 'title', 'section', 'text', 'html_tag'],
    num_rows: 1339236
})


In [7]:
#フィルタリング後の段落データの内容を確認する
pprint(paragraph_dataset[0])
pprint(paragraph_dataset[1])

{'html_tag': 'p',
 'id': '5-89167474-0',
 'pageid': 5,
 'paragraph_index': 0,
 'revid': 89167474,
 'section': '__LEAD__',
 'text': 'アンパサンド(&, 英語: '
         'ampersand)は、並立助詞「...と...」を意味する記号である。ラテン語で「...と...」を表す接続詞 "et" '
         'の合字を起源とする。現代のフォントでも、Trebuchet MS など一部のフォントでは、"et" '
         'の合字であることが容易にわかる字形を使用している。',
 'title': 'アンパサンド'}
{'html_tag': 'p',
 'id': '10-94194440-0',
 'pageid': 10,
 'paragraph_index': 0,
 'revid': 94194440,
 'section': '__LEAD__',
 'text': '言語(げんご)は、狭義には「声による記号の体系」をいう。',
 'title': '言語'}


#### トークナイザとモデルの準備

In [8]:
from google.colab import drive
drive.mount("drive")

Mounted at drive


In [10]:
!cp -r drive/MyDrive/llm-book/outputs_unsup_simcse .

In [15]:
from transformers import AutoModel, AutoTokenizer

# ディスクに保存された教師なしSimCSEのトークナイザとエンコーダを読み込む
model_path = "outputs_unsup_simcse/encoder"
tokenizer = AutoTokenizer.from_pretrained(model_path)
encoder = AutoModel.from_pretrained(model_path)

In [16]:
# 読み込んだモデルをGPUのメモリに移動させる
device = "cuda:0"
encoder = encoder.to(device)

#### モデルによる埋め込みの計算

In [19]:
! pip list  | grep -i torch

torch                                    2.10.0+cu128
torchao                                  0.10.0
torchaudio                               2.10.0+cu128
torchcodec                               0.10.0+cu128
torchdata                                0.11.0
torchsummary                             1.5.1
torchtune                                0.6.1
torchvision                              0.25.0+cu128


In [20]:
import numpy as np
import torch
import torch.nn.functional as F

def embed_texts(texts: list[str]) -> np.ndarray:
    """SimCSEのモデルを用いてテキストの埋め込みを作成"""
    # テキストにトークナイザを適用
    tokenized_texts = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    # トークナイズされたテキストをベクトルに変換
    # 勾配計算の無効化と自動混合精度を使用することで高速化
    with torch.inference_mode():  # 勾配計算の無効化（メモリ削減・高速化）
        with torch.amp.autocast("cuda"):  # 自動混合精度: float16/32の自動切り替え（GPU高速化・メモリ削減）
            encoded_texts = encoder(
                **tokenized_texts
            ).last_hidden_state[:,0]
    
    # ベクトルをNumPyのarrayに変換
    emb = encoded_texts.cpu().numpy().astype(np.float32)
    # ベクトルのノルムが1になるように正規化
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    return emb

In [21]:
# 段落データのすべての事例に埋め込みを付与する
paragraph_dataset = paragraph_dataset.map(
    lambda examples: {
        "embeddings": list(embed_texts(examples["text"]))
    },
    batched=True,
)

Parameter 'function'=<function <lambda> at 0x7b6cfc24ec00> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/1339236 [00:00<?, ? examples/s]

In [22]:
# 埋め込みを付与した段落データの形式と事例数を確認する
print(paragraph_dataset)

Dataset({
    features: ['id', 'pageid', 'revid', 'paragraph_index', 'title', 'section', 'text', 'html_tag', 'embeddings'],
    num_rows: 1339236
})


In [23]:
# 埋め込みを計算した段落データの内容を確認する
pprint(paragraph_dataset[0])

{'embeddings': [-0.04534943401813507,
                -0.01598762348294258,
                -0.060724347829818726,
                0.02832106128334999,
                -0.05233500152826309,
                -0.008481170050799847,
                -0.010506700724363327,
                0.04044181853532791,
                -0.02278558537364006,
                0.009886723011732101,
                0.05444807931780815,
                -0.05892454460263252,
                -0.07899744808673859,
                -0.04811432957649231,
                -0.006294048856943846,
                -0.013004260137677193,
                0.01203396450728178,
                -0.021361542865633965,
                -0.07127038389444351,
                -0.014846501871943474,
                0.009786671958863735,
                -0.02107825316488743,
                -0.029211383312940598,
                -0.0027480171993374825,
                0.049278054386377335,
                0.001363141112960875,
      

In [24]:
# 埋め込みを付与した段落データをデイスクに保存する
paragraph_dataset.save_to_disk(
    "outputs_unsup_simcse/embedded_paragraphs"
)

Saving the dataset (0/10 shards):   0%|          | 0/1339236 [00:00<?, ? examples/s]

#### Google Driveへの保存

In [25]:
from google.colab import drive
drive.mount("drive")

Drive already mounted at drive; to attempt to forcibly remount, call drive.mount("drive", force_remount=True).


In [26]:
# 保存された段落データをGoogleドライブのフォルダにコピーする
!cp -r outputs_unsup_simcse/embedded_paragraphs drive/MyDrive/llm-book/outputs_unsup_simcse

#### `Faiss`による最近傍探索を試す

In [28]:
import faiss

# ベクトルの次元数をエンコーダの設定値から取り出す
emb_dim = encoder.config.hidden_size
# ベクトルの図減数を指定して空のFaissインデックスを作成する
index = faiss.IndexFlatIP(emb_dim)
# 段落データの"embeddings"フィールドのベクトルからFaissインデックスを構築する
paragraph_dataset.add_faiss_index("embeddings", custom_index=index)

  0%|          | 0/1340 [00:00<?, ?it/s]

Dataset({
    features: ['id', 'pageid', 'revid', 'paragraph_index', 'title', 'section', 'text', 'html_tag', 'embeddings'],
    num_rows: 1339236
})

In [34]:
query_text = "日本語は、主に日本で話されている言語である"

# 最近傍探索を実行し、類似度上位10件の事例とスコアを取得する
scores, retrived_exaplmes = paragraph_dataset.get_nearest_examples(
    "embeddings", embed_texts([query_text])[0], k=10
)
# 取得した事例の内容をスコアとともに表示する
titles = retrived_exaplmes["title"]
texts = retrived_exaplmes["text"]
for score, title, text in zip(scores, titles, texts):
    print(f"{score} | {title} | {text}")

0.942046046257019 | トン普通語 | トン普通語(トンふつうご、トンふつご)とは、奄美語(特に奄美大島方言)の母語話者が標準日本語を第二言語として習得しようとした際に生成された中間言語(クレオール言語)と呼べる言語体系である。「トン」とは奄美語でサツマイモの意味。
0.9364888668060303 | イラクの言語 | イラクの言語(イラクのげんご)では、イラク国内で使用される言語について説明する。イラクでは多数の言語が使用されるが、最も広く話されるのはアラビア語イラク方言である。
0.9331420660018921 | トク・ピシン | トク・ピシン(Tok Pisin、トク・ピジンやネオ・メラネシア語とも)とは、英語を土台としたクレオール言語の一つであり、パプアニューギニアの公用語の一つである。約120万人がトク・ピシンを第一言語として使用し、300万人から400万人程度が第二言語として使用していると見られている。
0.9321753978729248 | 在日朝鮮語 | 在日朝鮮語(ざいにちちょうせんご、朝鮮語: 재일조선어)または在日韓国語(ざいにちかんこくご、재일한국어)とは、在日韓国・朝鮮人によって話される、朝鮮語の言語変種(language variety)のことを指す。後述するように話者による変異がある。
0.9311325550079346 | ビスラマ語 | ビスラマ語(ビスラマご、Bislama)は、メラネシア・ピジンに分類される一言語。バヌアツ共和国の公用語。英語とフランス語が交じり合い、変化して生まれた言語。正書法はまだ確立していない。
0.9306041598320007 | パイワン語 | パイワン語(パイワンご、パイワン語:pinaiwanan、中国語:排湾語)とは、主に台湾の先住民族の一つであるパイワン族によって話される言語である。オーストロネシア語族に属し、近代まで文字は持たなかったが、近代になってローマ字で表記されるようになり、正書法が定められている。
0.9300985336303711 | 日本手話語族 | 日本手話語族(にほんしゅわごぞく、Japanese Sign Language (JSL) family)は手話の語族である。日本手話、韓国手話、台湾手話が含まれる。これら3種の手話の相互間のコミ

#### indexを確認

In [36]:
print(f"次元数: {index.d}")
print(f"ベクトル数: {index.ntotal}")
print(f"index種類: {type(index).__name__}")

# 全ベクトルを取り出す
vectors = index.reconstruct_n(0, index.ntotal)
print(vectors.shape)  # (ntotal, d)


次元数: 768
ベクトル数: 1339236
index種類: IndexFlatIP
(1339236, 768)


In [37]:
! pip install pandas

In [41]:
import numpy as np
import pandas as pd
import plotly.express as px
import umap

# ① indexから全ベクトルを取り出す
vectors = index.reconstruct_n(0, index.ntotal)

# ② サンプリング（5000件くらいが快適）
n_sample = 5000
if index.ntotal > n_sample:
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(index.ntotal, size=n_sample, replace=False)
    vectors = vectors[sample_idx]
    texts = [paragraph_dataset[int(i)]["text"] for i in sample_idx]
else:
    texts = list(paragraph_dataset["text"])

# ③ UMAPで2次元に圧縮
coords = umap.UMAP(n_components=2, random_state=42).fit_transform(vectors)

# ④ ホバー用にテキスト短縮
short_texts = [t[:80] + ("..." if len(t) > 80 else "") for t in texts]

# ⑤ プロット作成
df = pd.DataFrame({
    "x": coords[:, 0],
    "y": coords[:, 1],
    "text": short_texts,
})

fig = px.scatter(
    df, x="x", y="y",
    hover_data={"text": True, "x": False, "y": False},
    opacity=0.6,
    title=f"SimCSE embeddings (UMAP 2D, n={len(df)})",
)
fig.update_traces(marker=dict(size=4))

# ⑥ HTMLとして書き出し（include_plotlyjsで自己完結化）
fig.write_html(
    "faiss.html",
    include_plotlyjs="cdn",  # JSはCDNから読み込み (ファイル軽量化)
    full_html=True,
)

# Colab上で表示もしたい場合
fig.show()

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

